# PDD Mobility Scanner — Format trail_data.csv for Dashboard

Convert the raw V4 firmware output (`trail_data.csv` + `trail_video.mp4` + `video_sync.csv`) into the dashboard-compatible CSV format with `wp`, `image_file`, `surface_type`, `obstacle`, and `substructure` columns.

**Mapping rules:**
- `wp` — 5-second waypoint windows derived from video time; one waypoint = ~125 sensor rows at 25Hz
- `image_file` — the **first** mp4 frame inside each waypoint, saved as `img_NNNN.jpg`; every row in the waypoint references the same filename
- `surface_type` / `obstacle` / `substructure` — placeholder strings (`Not available` / `Not assessed`); obstacle auto-detection via YOLO is a later step

## Step 1: Install dependencies

In [ ]:
import pandas as pd
import numpy as np
import cv2
import os
import shutil
from google.colab import files

## Step 2: Upload files

Upload all three from the SD card: `trail_data.csv`, `trail_video.mp4`, `video_sync.csv`.

In [ ]:
uploaded = files.upload()

csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]
trail_csv = next(f for f in csv_files if 'sync' not in f.lower())
sync_csv  = next(f for f in csv_files if 'sync' in f.lower())
video_file = next(f for f in uploaded.keys() if f.endswith('.mp4'))

print(f'Trail CSV: {trail_csv}')
print(f'Sync CSV:  {sync_csv}')
print(f'Video:     {video_file}')

## Step 3: Load sync info and sensor data

`video_sync.csv` carries the `rec_start_ms` reference so we can convert each sensor row's `ms` into video time.

In [ ]:
sync = pd.read_csv(sync_csv)
rec_start_ms = int(sync['rec_start_ms'].iloc[0])
print(f'rec_start_ms = {rec_start_ms}')

df = pd.read_csv(trail_csv)
df['video_sec'] = (df['ms'] - rec_start_ms) / 1000.0
print(f'Loaded {len(df)} sensor rows, video duration {df["video_sec"].max():.1f}s')

## Step 4: Assign waypoints (5-second windows)

`wp = floor(video_sec / 5)`, clipped at 0 so any sensor rows logged before the mp4 started are folded into waypoint 0.

In [ ]:
WAYPOINT_SEC = 5
df['wp'] = np.maximum(0, np.floor(df['video_sec'] / WAYPOINT_SEC).astype(int))
n_waypoints = df['wp'].nunique()
print(f'{n_waypoints} waypoints over {len(df)} rows '
      f'(avg {len(df) / n_waypoints:.0f} rows per waypoint)')

## Step 5: Extract one frame per waypoint

For each waypoint, pick the **first** mp4 frame at or after `wp * 5` seconds. Save as `img_NNNN.jpg` and record the filename so every sensor row in that waypoint can reference it.

In [ ]:
out_dir = 'output'
img_dir = os.path.join(out_dir, 'images')
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(img_dir)

cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {total_frames} frames at {fps:.2f} FPS')

wp_to_image = {}
for wp in sorted(df['wp'].unique()):
    target_sec = wp * WAYPOINT_SEC
    target_frame = min(int(round(target_sec * fps)), total_frames - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()
    if not ret:
        # Fall back to the previous waypoint's image if the video ended early
        wp_to_image[wp] = wp_to_image.get(wp - 1, '')
        continue
    fname = f'img_{wp:04d}.jpg'
    cv2.imwrite(os.path.join(img_dir, fname), frame)
    wp_to_image[wp] = fname

cap.release()
print(f'Wrote {len([v for v in wp_to_image.values() if v])} images to {img_dir}')

## Step 6: Build the formatted CSV

Add `image_file` (per-waypoint), placeholder annotation columns, drop the helper `video_sec`, and reorder to match the dashboard schema exactly.

In [ ]:
df['image_file']    = df['wp'].map(wp_to_image)
df['surface_type']  = 'Not available'
df['obstacle']      = 'Not assessed'
df['substructure']  = 'Not assessed'

TARGET_COLUMNS = [
    'ms', 'utc', 'wp',
    'ax', 'ay', 'az',
    'gvx', 'gvy', 'gvz',
    'gx', 'gy', 'gz',
    'qw', 'qx', 'qy', 'qz',
    'mx', 'my', 'mz',
    'tof_mm', 'tof_status',
    'lat', 'lng', 'alt', 'speed', 'hdg', 'sats', 'hdop',
    'cal_sys', 'cal_gyro', 'cal_accel', 'cal_mag',
    'image_file', 'surface_type', 'obstacle', 'substructure',
]

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
assert not missing, f'Missing columns from input: {missing}'

out_df = df[TARGET_COLUMNS]
out_csv = os.path.join(out_dir, 'trail_data.csv')
out_df.to_csv(out_csv, index=False)

print(f'Wrote {out_csv} ({len(out_df)} rows, {len(out_df.columns)} columns)')
out_df.head()

## Step 7: Download bundle

Zip the formatted CSV together with the per-waypoint images and download as a single file.

In [ ]:
bundle = shutil.make_archive('trail_data_formatted', 'zip', out_dir)
print(f'Bundle: {bundle}')
files.download(bundle)